In [42]:
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetV2S
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report

import seaborn as sns
import matplotlib.pyplot as plt

In [43]:
IMG_SIZE = 224
BATCH_SIZE = 32

tabular_dir = "training dataset/tabular"
image_dir = "training dataset/image"

train_df = pd.read_csv(os.path.join(tabular_dir, "train_split.csv"))
val_df = pd.read_csv(os.path.join(tabular_dir, "val_split.csv"))
test_df = pd.read_csv(os.path.join(tabular_dir, "test_split.csv"))

view_cols = [c for c in train_df.columns if "View" in c and "Path" in c]

In [44]:
def finalize_research_model(inputs, x, model_name):
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

In [45]:
def build_mobilenet_v2():
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "MobileNetV2")

def build_resnet_50():
    base = ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "ResNet50")

def build_efficientnet_v2s():
    base = EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "EfficientNetV2S")

In [46]:
def data_generator(df):
    for idx, row in df.iterrows():
        if row.get('Has_Images', 0) == 1:
            label = row['Diagnosis']

            for col in view_cols:
                p = row[col]
                if pd.notna(p) and p != "MISSING_IMAGE":

                    p = p.replace('\\', os.sep).replace('/', os.sep)
                    full_path = os.path.join(image_dir, p)

                    img = cv2.imread(full_path)
                    if img is not None:
                        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                        img = img / 255.0

                        yield img.astype(np.float32), np.int32(label)

In [47]:
def get_ds(df, shuffle=False):
    ds = tf.data.Dataset.from_generator(
        lambda: data_generator(df),
        output_signature=(
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
            tf.TensorSpec((), tf.int32)
        )
    )

    if shuffle:
        ds = ds.shuffle(500)

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [48]:
def build_model(name):
    if name == "MobileNetV2":
        return build_mobilenet_v2()
    elif name == "ResNet50":
        return build_resnet_50()
    elif name == "EfficientNetV2S":
        return build_efficientnet_v2s()

In [49]:
def evaluate_patient_level(model, df):
    patient_preds = defaultdict(list)
    patient_labels = {}

    for idx, row in df.iterrows():
        if row.get('Has_Images', 0) == 1:
            label = row['Diagnosis']

            for col in view_cols:
                p = row[col]
                if pd.notna(p) and p != "MISSING_IMAGE":

                    p = p.replace('\\', os.sep).replace('/', os.sep)
                    full_path = os.path.join(image_dir, p)

                    img = cv2.imread(full_path)
                    if img is not None:
                        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
                        img = np.expand_dims(img, axis=0)

                        pred = model.predict(img, verbose=0)[0][0]

                        patient_preds[idx].append(pred)
                        patient_labels[idx] = label

    y_true, y_pred = [], []

    for pid in patient_preds:
        avg_pred = np.mean(patient_preds[pid])
        final_pred = 1 if avg_pred > 0.5 else 0

        y_true.append(patient_labels[pid])
        y_pred.append(final_pred)

    return y_true, y_pred

In [50]:
def run_benchmark():
    models = ["MobileNetV2", "ResNet50", "EfficientNetV2S"]
    results = {}

    train_ds = get_ds(train_df, shuffle=True)
    val_ds = get_ds(val_df)

    for m_name in models:
        print(f"\n🚀 Training {m_name}")
        model = build_model(m_name)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=10,
            callbacks=[
                EarlyStopping(monitor='val_auc', patience=3, mode='max', restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2)
            ]
        )

        # 🔥 Patient-level evaluation
        y_true, y_pred = evaluate_patient_level(model, test_df)

        cm = confusion_matrix(y_true, y_pred)

        print(f"\n📊 Confusion Matrix - {m_name}")
        print(cm)

        # Heatmap
        plt.figure()
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive']
        )
        plt.title(f'Confusion Matrix - {m_name}')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.show()

        print(f"\n📋 Classification Report - {m_name}")
        print(classification_report(y_true, y_pred))

        results[m_name] = {
            "Accuracy": round(np.mean(np.array(y_true) == np.array(y_pred)), 4),
            "TP": int(cm[1][1]),
            "TN": int(cm[0][0]),
            "FP": int(cm[0][1]),
            "FN": int(cm[1][0])
        }

    return pd.DataFrame(results).T

In [ ]:
performance = run_benchmark()
print("\n📊 Final Results:")
print(performance)


🚀 Training MobileNetV2
Epoch 1/10
     38/Unknown 46s 639ms/step - accuracy: 0.6827 - auc: 0.4740 - loss: 1.0791

C:\Users\Trinabh Mitra\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


38/38 ━━━━━━━━━━━━━━━━━━━━ 55s 880ms/step - accuracy: 0.7042 - auc: 0.5257 - loss: 1.0570 - val_accuracy: 0.8470 - val_auc: 0.6905 - val_loss: 0.8502 - learning_rate: 1.0000e-04
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 36s 744ms/step - accuracy: 0.7730 - auc: 0.6621 - loss: 0.9456 - val_accuracy: 0.8415 - val_auc: 0.7054 - val_loss: 0.8636 - learning_rate: 1.0000e-04
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 33s 731ms/step - accuracy: 0.7697 - auc: 0.6623 - loss: 0.9497 - val_accuracy: 0.8415 - val_auc: 0.7193 - val_loss: 0.8663 - learning_rate: 1.0000e-04
Epoch 4/10
